In [7]:
import pandas as pd
import os
import pandas as pd

df = pd.read_csv("dataset/indian_movies.csv")

In [8]:
movies = df[['Movie Name',
             'Genre',
             'Rating(10)',
             'Language',
             'Year',
             'Votes']].copy()

movies = movies.rename(columns={
    'Movie Name': 'Title',
    'Rating(10)': 'Rating'
})

movies = movies.dropna()

movies['Rating'] = pd.to_numeric(
    movies['Rating'],
    errors='coerce'
)

movies['Year'] = pd.to_numeric(
    movies['Year'],
    errors='coerce'
)

movies = movies.dropna()

In [9]:
file_path = os.path.join("dataset", "indian_movies.csv")
df = pd.read_csv(file_path)

df.head()

,ID,Movie Name,Year,Timing(min),Rating(10),Votes,Genre,Language
0,tt0398974,Dr. Shaitan,1960,-,-,-,-,hindi
1,tt1702558,Nadir Khan,1968,-,-,-,-,urdu
2,tt0493437,Apna Sapna Money Money,2006,134 min,5.3,"1,892","Comedy, Musical, Romance",hindi
3,tt0273405,Aag Aur Sholay,1987,-,2.2,20,-,urdu
4,tt0049595,Parivar,1956,-,7.4,21,"Comedy, Drama, Family",hindi


In [11]:
movies = movies.head(5000)

print(movies.shape)

(5000, 6)


In [12]:
movies['tags'] = (
    movies['Title'].astype(str)
    + " "
    + movies['Genre'].astype(str)
)

In [13]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(
    max_features=3000,
    stop_words='english'
)

vectors = cv.fit_transform(
    movies['tags']
).toarray()

print(vectors.shape)

(5000, 3000)


In [14]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(vectors)

print(similarity.shape)

(5000, 5000)


In [16]:
print("Movies Shape:", movies.shape)
print("Vectors Shape:", vectors.shape)

Movies Shape: (5000, 7)
Vectors Shape: (5000, 3000)


In [17]:
movies = movies.head(5000)

movies['tags'] = (
    movies['Title'].astype(str)
    + " "
    + movies['Genre'].astype(str)
)

from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(max_features=3000, stop_words='english')

vectors = cv.fit_transform(movies['tags']).toarray()

print(vectors.shape)

(5000, 3000)


In [15]:
def recommend(movie):

    movie = movie.lower()

    matches = movies[
        movies['Title']
        .str.lower()
        .str.contains(movie, na=False)
    ]

    if matches.empty:
        return "Movie Not Found"

    index = matches.index[0]

    distances = similarity[index]

    movie_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:6]

    recommendations = []

    for i in movie_list:
        recommendations.append(
            movies.iloc[i[0]]['Title']
        )

    return recommendations

In [19]:
recommend("Apna Sapna Money Money")

NameError: name 'recommend' is not defined

In [20]:
movies['Title'].head(20)

2      Apna Sapna Money Money
3              Aag Aur Sholay
4                     Parivar
6      Jacqueline I Am Coming
7              A Mighty Heart
9                  Raktalekha
10          Veedevadandi Babu
12          Da Tang Xuan Zang
15    Amman Koil Kizhakkaalae
16                     Shatru
19                    Tukaram
20        Ramante Edenthottam
28                    Dracula
30                       Geet
33           Nuvvu Vasthavani
35                  Madadgaar
37              Milan Ki Raat
38            Bhool Bhulaiyan
39       Saajan Chale Sasural
40                    Khaleja
Name: Title, dtype: str

In [23]:
def mood_recommend(mood):

    mood = mood.lower()

    if mood == "happy":
        data = movies[
            movies['Genre']
            .str.contains("Comedy|Family",
                          case=False,
                          na=False)
        ]

    elif mood == "sad":
        data = movies[
            movies['Genre']
            .str.contains("Drama",
                          case=False,
                          na=False)
        ]

    elif mood == "romantic":
        data = movies[
            movies['Genre']
            .str.contains("Romance",
                          case=False,
                          na=False)
        ]

    elif mood == "action":
        data = movies[
            movies['Genre']
            .str.contains("Action",
                          case=False,
                          na=False)
        ]

    else:
        data = movies.sample(5)

    return data[['Title', 'Genre', 'Rating']].head(10)

In [24]:
mood_recommend("happy")

,Title,Genre,Rating
2,Apna Sapna Money Money,"Comedy, Musical, Romance",5.3
4,Parivar,"Comedy, Drama, Family",7.4
10,Veedevadandi Babu,Comedy,5.6
20,Ramante Edenthottam,"Comedy, Drama, Romance",6.0
33,Nuvvu Vasthavani,"Family, Romance",6.6
38,Bhool Bhulaiyan,Comedy,6.0
39,Saajan Chale Sasural,"Comedy, Drama",6.0
40,Khaleja,"Action, Comedy, Fantasy",7.6
54,Gopi Kishan,"Action, Comedy, Drama",5.5
55,Nigahen: Nagina Part II,"Drama, Family, Fantasy",5.2


In [29]:
def language_filter(language):

    data = movies[
        movies['Language']
        .str.lower()
        == language.lower()
    ]

    return data[
        ['Title',
         'Language',
         'Genre',
         'Rating']
    ].head(20)

In [30]:
language_filter("hindi")

,Title,Language,Genre,Rating
2,Apna Sapna Money Money,hindi,"Comedy, Musical, Romance",5.3
4,Parivar,hindi,"Comedy, Drama, Family",7.4
6,Jacqueline I Am Coming,hindi,Drama,7.9
28,Dracula,hindi,Horror,4.1
30,Geet,hindi,"Action, Crime, Musical",7.1
35,Madadgaar,hindi,"Action, Crime, Drama",5.4
37,Milan Ki Raat,hindi,"Drama, Romance",4.4
38,Bhool Bhulaiyan,hindi,Comedy,6.0
39,Saajan Chale Sasural,hindi,"Comedy, Drama",6.0
53,Time to Dance,hindi,"Musical, Romance",1.3


In [27]:
language_filter("tamil")

,Title,Language,Genre,Rating
15,Amman Koil Kizhakkaalae,tamil,Drama,6.9
57,Pagaivan,tamil,"Action, Comedy",5.2
66,Vidiyum Munn,tamil,Thriller,7.4
110,Puthayal,tamil,-,5.9
124,Kanthaswamy,tamil,"Action, Crime",5.0
125,Vishwa Thulasi,tamil,"Drama, Musical, Romance",6.1
132,Maithaanam,tamil,"Drama, Family, Mystery",5.9
133,Sons of Ram,tamil,"Animation, Drama, Musical",6.6
170,Deiva Thirumagal,tamil,Drama,8.1
179,Talentime,tamil,Drama,6.8


In [31]:
language_filter("hindi")

,Title,Language,Genre,Rating
2,Apna Sapna Money Money,hindi,"Comedy, Musical, Romance",5.3
4,Parivar,hindi,"Comedy, Drama, Family",7.4
6,Jacqueline I Am Coming,hindi,Drama,7.9
28,Dracula,hindi,Horror,4.1
30,Geet,hindi,"Action, Crime, Musical",7.1
35,Madadgaar,hindi,"Action, Crime, Drama",5.4
37,Milan Ki Raat,hindi,"Drama, Romance",4.4
38,Bhool Bhulaiyan,hindi,Comedy,6.0
39,Saajan Chale Sasural,hindi,"Comedy, Drama",6.0
53,Time to Dance,hindi,"Musical, Romance",1.3


In [32]:
def rating_filter(min_rating):

    data = movies[
        movies['Rating'] >= min_rating
    ]

    return data[
        ['Title',
         'Genre',
         'Language',
         'Rating']
    ].sort_values(
        by='Rating',
        ascending=False
    ).head(20)

In [33]:
rating_filter(7)

,Title,Genre,Language,Rating
7812,Ilu-Ilu,Drama,marathi,10.0
5622,Domestic Dialogues,Drama,malayalam,9.7
4263,Kunku Zale Vairi,Drama,marathi,9.6
9312,Kanku,Drama,gujarati,9.6
5205,Fireflies-Jonaki Porua,Drama,assamese,9.6
4142,The Mastermind: Jinda Sukha,"Action, Biography, History",punjabi,9.6
6420,Ranganayaki,Drama,kannada,9.5
5340,Qawwali - Music of the Mystics,Music,oriya,9.5
11316,Taawdo the Sunlight,Drama,rajastani,9.5
5042,Akhri Chattan,History,urdu,9.5


In [34]:
rating_filter(8)

,Title,Genre,Language,Rating
7812,Ilu-Ilu,Drama,marathi,10.0
5622,Domestic Dialogues,Drama,malayalam,9.7
5205,Fireflies-Jonaki Porua,Drama,assamese,9.6
4142,The Mastermind: Jinda Sukha,"Action, Biography, History",punjabi,9.6
4263,Kunku Zale Vairi,Drama,marathi,9.6
9312,Kanku,Drama,gujarati,9.6
5042,Akhri Chattan,History,urdu,9.5
6420,Ranganayaki,Drama,kannada,9.5
5340,Qawwali - Music of the Mystics,Music,oriya,9.5
11316,Taawdo the Sunlight,Drama,rajastani,9.5


In [35]:
rating_filter(9)

,Title,Genre,Language,Rating
7812,Ilu-Ilu,Drama,marathi,10.0
5622,Domestic Dialogues,Drama,malayalam,9.7
4142,The Mastermind: Jinda Sukha,"Action, Biography, History",punjabi,9.6
5205,Fireflies-Jonaki Porua,Drama,assamese,9.6
4263,Kunku Zale Vairi,Drama,marathi,9.6
9312,Kanku,Drama,gujarati,9.6
6420,Ranganayaki,Drama,kannada,9.5
11316,Taawdo the Sunlight,Drama,rajastani,9.5
5340,Qawwali - Music of the Mystics,Music,oriya,9.5
5042,Akhri Chattan,History,urdu,9.5


In [36]:
def year_filter(min_year):

    data = movies[
        movies['Year'] >= min_year
    ]

    return data[
        ['Title',
         'Year',
         'Genre',
         'Language',
         'Rating']
    ].sort_values(
        by='Year',
        ascending=False
    ).head(20)

In [37]:
year_filter(2020)

,Title,Year,Genre,Language,Rating
53,Time to Dance,2021.0,"Musical, Romance",hindi,1.3
93,Preetam,2021.0,Romance,marathi,9.0
231,Tribhanga,2021.0,"Drama, Family",hindi,6.1
508,Tuesdays and Fridays,2021.0,"Comedy, Musical, Romance",hindi,4.5
1063,Ninnila Ninnila,2021.0,"Comedy, Romance",telugu,7.5
512,Father Chitti Umaa Kaarthik,2021.0,Drama,telugu,6.5
1753,Roohi,2021.0,"Comedy, Horror",hindi,4.2
2858,Lollipops,2021.0,Thriller,tamil,3.8
2392,Pagglait,2021.0,"Comedy, Drama",hindi,7.0
2851,Madam Chief Minister,2021.0,Drama,hindi,4.6


In [38]:
year_filter(2015)

,Title,Year,Genre,Language,Rating
11386,Kartha,2021.0,Drama,kannada,9.4
231,Tribhanga,2021.0,"Drama, Family",hindi,6.1
11374,Ramarjuna,2021.0,Action,kannada,8.5
9523,Inspector Vikram,2021.0,"Action, Comedy, Crime",kannada,5.6
4107,April 28 Em Jarigindi,2021.0,"Drama, Horror, Thriller",telugu,5.3
7414,Story Night,2021.0,"Short, Horror",urdu,8.5
9756,Mangalavara Rajaadina,2021.0,Drama,kannada,8.0
9722,A1 Express,2021.0,Drama,telugu,6.8
3501,Varthamanam,2021.0,Drama,malayalam,8.2
3561,Thinkalazhcha Nishchayam,2021.0,"Comedy, Drama, Romance",malayalam,9.4


In [39]:
year_filter(2018)

,Title,Year,Genre,Language,Rating
93,Preetam,2021.0,Romance,marathi,9.0
231,Tribhanga,2021.0,"Drama, Family",hindi,6.1
7330,Sashi,2021.0,Romance,telugu,4.4
508,Tuesdays and Fridays,2021.0,"Comedy, Musical, Romance",hindi,4.5
512,Father Chitti Umaa Kaarthik,2021.0,Drama,telugu,6.5
7097,Amanuda,2021.0,"Horror, Mystery",tamil,8.7
7149,Anbirkiniyal,2021.0,"Drama, Thriller",tamil,6.7
7848,Master,2021.0,"Action, Thriller",tamil,7.0
6678,Janowar,2021.0,"Crime, Drama, Thriller",bengali,8.0
6803,Sultan,2021.0,"Action, Drama",tamil,6.2


In [40]:
def advanced_filter(language=None,
                    min_rating=None,
                    year=None):

    data = movies.copy()

    if language:
        data = data[
            data['Language']
            .str.lower()
            == language.lower()
        ]

    if min_rating:
        data = data[
            data['Rating']
            >= min_rating
        ]

    if year:
        data = data[
            data['Year']
            >= year
        ]

    return data[
        ['Title',
         'Language',
         'Genre',
         'Rating',
         'Year']
    ].sort_values(
        by='Rating',
        ascending=False
    ).head(20)

In [41]:
advanced_filter(
    language="hindi",
    min_rating=7,
    year=2015
)

,Title,Language,Genre,Rating,Year
9380,Ashok Vatika,hindi,Drama,9.3,2018.0
4037,Yug the law of karma,hindi,Drama,9.0,2021.0
11228,Salt Bridge,hindi,Drama,8.9,2017.0
9846,Machaan,hindi,"Drama, Romance",8.9,2021.0
6659,B for Bundelkhand,hindi,Drama,8.8,2017.0
9891,Colour photo,hindi,Comedy,8.7,2018.0
4867,Yeh Suhaagraat Impossible,hindi,Comedy,8.7,2019.0
11090,Little Baby,hindi,Family,8.7,2019.0
10295,Officer Arjun Singh IPS,hindi,"Action, Drama",8.6,2019.0
500,Kuch Der Aur,hindi,Drama,8.5,2018.0


In [42]:
advanced_filter(
    language="tamil",
    min_rating=8,
    year=2020
)

,Title,Language,Genre,Rating,Year
7097,Amanuda,tamil,"Horror, Mystery",8.7,2021.0
4200,Mandela,tamil,"Comedy, Drama",8.5,2021.0


In [43]:
advanced_filter(
    language="telugu",
    min_rating=7
)

,Title,Language,Genre,Rating,Year
461,Madhanam,telugu,"Comedy, Romance",8.9,2019.0
4006,Aha Naa Pellanta,telugu,Comedy,8.9,1987.0
1218,Thirugubatu,telugu,-,8.8,1985.0
11427,99 Songs,telugu,"Musical, Romance",8.8,2019.0
8955,O Manishi Thirigi Chudu,telugu,-,8.8,1976.0
452,Sankarabharanam,telugu,"Drama, Music, Romance",8.7,1980.0
8869,Illaliko Pariksha,telugu,-,8.7,1985.0
7409,Seeta Ramulu,telugu,-,8.7,1980.0
2814,Kadaladu Vadaladu,telugu,-,8.7,1969.0
4829,Nuvvu Naaku Nachchav,telugu,"Comedy, Romance",8.7,2001.0


In [44]:
def top_movies():

    return movies[
        ['Title',
         'Genre',
         'Language',
         'Rating']
    ].sort_values(
        by='Rating',
        ascending=False
    ).head(20)

In [45]:
top_movies()

,Title,Genre,Language,Rating
7812,Ilu-Ilu,Drama,marathi,10.0
5622,Domestic Dialogues,Drama,malayalam,9.7
9312,Kanku,Drama,gujarati,9.6
4263,Kunku Zale Vairi,Drama,marathi,9.6
4142,The Mastermind: Jinda Sukha,"Action, Biography, History",punjabi,9.6
5205,Fireflies-Jonaki Porua,Drama,assamese,9.6
6420,Ranganayaki,Drama,kannada,9.5
5042,Akhri Chattan,History,urdu,9.5
11316,Taawdo the Sunlight,Drama,rajastani,9.5
5340,Qawwali - Music of the Mystics,Music,oriya,9.5


In [46]:
movies['Votes'].head(10)

2      1,892
3         20
4         21
6         16
7     26,885
9         12
10       218
12       379
15        63
16        11
Name: Votes, dtype: str

In [47]:
movies['Votes'] = (
    movies['Votes']
    .astype(str)
    .str.replace(',', '')
)

movies['Votes'] = pd.to_numeric(
    movies['Votes'],
    errors='coerce'
)

movies['Votes'] = movies['Votes'].fillna(0)

In [48]:
def trending_movies():

    return movies[
        ['Title',
         'Genre',
         'Language',
         'Votes',
         'Rating']
    ].sort_values(
        by='Votes',
        ascending=False
    ).head(20)

In [49]:
trending_movies()

,Title,Genre,Language,Votes,Rating
471,Iron Man,"Action, Adventure, Sci-Fi",urdu,954861,7.9
11493,Star Wars: Episode I - The Phantom Menace,"Action, Adventure, Fantasy",sanskrit,740452,6.5
2122,The Jungle Book,"Adventure, Drama, Family",oriya,261319,7.4
11368,The Jungle Book,"Adventure, Drama, Family",gujarati,261312,7.4
10499,Eastern Promises,"Action, Crime, Drama",urdu,230284,7.6
11438,The Darjeeling Limited,"Adventure, Comedy, Drama",punjabi,183367,7.2
7160,A Walk Among the Tombstones,"Action, Crime, Drama",oriya,113715,6.5
9257,Charlie Wilson's War,"Biography, Comedy, Drama",urdu,111934,7.0
4713,Chak De! India,"Drama, Family, Sport",hindi,74911,8.2
10051,Chungking Express,"Comedy, Crime, Drama",oriya,66061,8.1


In [50]:
def search_movies(query):

    results = movies[
        movies['Title']
        .str.contains(
            query,
            case=False,
            na=False
        )
    ]

    return results[
        ['Title',
         'Language',
         'Rating']
    ].head(10)

In [51]:
search_movies("bha")

,Title,Language,Rating
146,Bhagwaan Dada,hindi,4.3
231,Tribhanga,hindi,6.1
452,Sankarabharanam,telugu,8.7
487,Sampoorna Mahabharat,gujarati,7.8
719,Bhanta,kannada,7.1
786,Tobou Bhalobashi,bengali,3.6
1077,Bhoga Bhagyalu,telugu,4.9
1359,Sarabham,tamil,7.1
1675,Munnabhai S.S.C.,marathi,4.3
1733,Bhagaban Shrikrishna Chaitanya,bengali,8.0


In [52]:
search_movies("kal")

,Title,Language,Rating
176,Kala Dhanda Goray Log,hindi,5.7
201,Kalsandhya,assamese,7.8
589,Kalyug,hindi,7.8
722,Daal Me Kala,hindi,6.0
797,Rektha Sakshikal Zindabad,malayalam,5.8
972,Khaidi Kalidasu,telugu,8.3
995,Tatamma Kala,telugu,8.2
1196,Makkal Aatchi,tamil,7.1
1199,Muhtukkal Moonru,tamil,5.8
1667,Kunjattakilikal,malayalam,6.7


In [53]:
search_movies("master")

,Title,Language,Rating
4142,The Mastermind: Jinda Sukha,punjabi,9.6
7848,Master,tamil,7.0


In [54]:
def recommendation_reason(
    movie,
    mood,
    language
):

    return (
        f"Recommended because it is "
        f"similar to {movie}, "
        f"matches {mood} mood, "
        f"and is available in "
        f"{language} language."
    )

In [55]:
recommendation_reason(
    "Kuselan",
    "happy",
    "hindi"
)

'Recommended because it is similar to Kuselan, matches happy mood, and is available in hindi language.'

In [56]:
stats = {
    "Total Movies": len(movies),
    "Languages": movies['Language'].nunique(),
    "Genres": movies['Genre'].nunique(),
    "Average Rating":
        round(movies['Rating'].mean(), 2)
}

print(stats)

{'Total Movies': 5000, 'Languages': 19, 'Genres': 336, 'Average Rating': np.float64(6.22)}


In [57]:
watchlist = []

In [59]:
def add_to_watchlist(movie):

    watchlist.append(movie)

    return watchlist

In [60]:
add_to_watchlist("Kuselan")
add_to_watchlist("Bhool Bhulaiyan")

print(watchlist)

['Kuselan', 'Bhool Bhulaiyan']


In [61]:
print(movies.columns)


Index(['Title', 'Genre', 'Rating', 'Language', 'Year', 'Votes', 'tags'], dtype='str')


In [75]:
mood_mapping = {
    "Happy": ["Comedy"],
    "Sad": ["Drama"],
    "Romantic": ["Romance"],
    "Excited": ["Action", "Adventure"]
}

In [66]:
def filter_movies(mood, language, rating, year):

    filtered = df.copy()

    if mood:
        genres = mood_mapping.get(mood, [])

        filtered = filtered[
            filtered['Genre'].apply(
                lambda x: any(g.lower() in str(x).lower() for g in genres)
            )
        ]

    if language:
        filtered = filtered[
            filtered['Language'].str.lower() == language.lower()
        ]

    if rating:
        filtered = filtered[
            filtered['Rating(10)'] >= float(rating)
        ]

    if year:
        filtered = filtered[
            filtered['Year'] >= int(year)
        ]

    return filtered[['Movie Name','Genre','Rating(10)','Language','Year']]

In [74]:
result = filter_movies(
    mood="Excited",
    language="Hindi",
    rating=7,
    year=2020
)

print(result.head(10))

                              Movie Name  \
6193                            AK vs AK   
13156                    Run Zindagi Run   
13313  Ek Sainik - The Tale of a warrior   
13754                             Kaadan   
18295        Tanhaji: The Unsung Warrior   
22588                            Torbaaz   
30600                               Ludo   
32869                         8119 Miles   
40658                       Khuda Haafiz   
42708                              Taish   

                                       Genre  Rating(10) Language    Year  
6193       Action, Comedy, Crime                     7.0    hindi  2020.0  
13156    Action, Crime, Thriller                     7.5    hindi  2020.0  
13313              Action, Drama                     8.2    hindi  2020.0  
13754     Action, Drama, Fantasy                     7.0    hindi  2021.0  
18295   Action, Biography, Drama                     7.6    hindi  2020.0  
22588       Action, Drama, Sport                     7.0   

In [68]:
print(df['Rating(10)'].dtype)

str


In [69]:
df['Rating(10)'] = pd.to_numeric(
    df['Rating(10)'],
    errors='coerce'
)

In [70]:
df['Year'] = pd.to_numeric(
    df['Year'],
    errors='coerce'
)

In [71]:
print(df['Rating(10)'].dtype)
print(df['Year'].dtype)

float64
float64


In [73]:
result = filter_movies(
    mood="Excited",
    language="Hindi",
    rating=7,
    year=2020
)

print(result.head())
print(len(result))

                              Movie Name  \
6193                            AK vs AK   
13156                    Run Zindagi Run   
13313  Ek Sainik - The Tale of a warrior   
13754                             Kaadan   
18295        Tanhaji: The Unsung Warrior   

                                      Genre  Rating(10) Language    Year  
6193      Action, Comedy, Crime                     7.0    hindi  2020.0  
13156   Action, Crime, Thriller                     7.5    hindi  2020.0  
13313             Action, Drama                     8.2    hindi  2020.0  
13754    Action, Drama, Fantasy                     7.0    hindi  2021.0  
18295  Action, Biography, Drama                     7.6    hindi  2020.0  
11
